In [29]:
import pandas as pd
import os
from glob import glob
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import Input, Dense, Conv1D, MaxPooling1D, Flatten
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.metrics import CategoricalAccuracy
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import CategoricalAccuracy
from tensorflow.keras.regularizers import l2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import json
import random
from xgboost import XGBClassifier

# Loading the datframes from the CSVs

In [3]:
# read csv files as dataframes
embeddings = pd.read_csv('C:\\Users\\sigma\Documents\\GitHub\ASE_2024\\c_binary_classifiers\\labeled_hyperparameter_embedding.csv')
embeddings 

,model_file,embedding,label
0,7657_68692047_Hyperparameter_buggy.h5,"[0.16367369890213013, 0.23748232424259186, 0.0...",Hyperparameter
1,3639_50481178_correct.h5,"[0.18405327200889587, 0.2728056311607361, 0.10...",correct
2,4523_71351646_Hyperparameter_buggy.h5,"[0.17668721079826355, 0.2513545751571655, 0.11...",Hyperparameter
3,3969_61553510_correct.h5,"[0.16761009395122528, 0.2628805637359619, 0.10...",correct
4,670_48934338_Hyperparameter_buggy.h5,"[0.1766625940799713, 0.2602795362472534, 0.078...",Hyperparameter
...,...,...,...
8157,1919_46642627_correct.h5,"[0.19169928133487701, 0.2572789192199707, 0.09...",correct
8158,1415_77416892_correct.h5,"[0.16933181881904602, 0.2405780851840973, 0.11...",correct
8159,4554_71351646_Hyperparameter_buggy.h5,"[0.17193560302257538, 0.25609657168388367, 0.1...",Hyperparameter
8160,5707_73765263_Hyperparameter_buggy.h5,"[0.16586410999298096, 0.2581118941307068, 0.11...",Hyperparameter


In [4]:
# read csv files as dataframes
graphs = pd.read_csv('C:\\Users\\sigma\Documents\\GitHub\ASE_2024\\c_binary_classifiers\\labeled_hyperparameter_graph.csv')
graphs

,degree_centrality,closeness_centrality,betweenness_centrality,degree,transitivity,shortest_path_length,eigenvector_centrality,pagerank,hubs,authorities,k_core,density,label
0,"{'0': 1.0, '1': 1.0}","{'0': 0.0, '1': 1.0}","{'0': 0.0, '1': 0.0}","{'0': 1, '1': 1}",0,"{'0': {'0': 0, '1': 1}, '1': {'1': 0}}","{'0': 0.001410435832630306, '1': 0.99999900533...","{'0': 0.3508773619358619, '1': 0.649122638064138}","{'0': 1.0, '1': 0.0}","{'0': 0.0, '1': 1.0}","{'0': 1, '1': 1}",0.500000,Hyperparameter
1,"{'0': 0.2, '1': 0.4, '2': 0.4, '3': 0.4, '4': ...","{'0': 0.0, '1': 0.2, '2': 0.26666666666666666,...","{'0': 0.0, '1': 0.2, '2': 0.30000000000000004,...","{'0': 1, '1': 2, '2': 2, '3': 2, '4': 2, '5': 1}",0,"{'0': {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4,...","{'0': 1.800995096925549e-13, '1': 1.6641194695...","{'0': 0.0607158048829909, '1': 0.1123242344582...","{'0': 0.9783551287573491, '1': -0.006167686523...","{'0': -0.0, '1': 0.9783551287573491, '2': -0.0...","{'0': 1, '1': 1, '2': 1, '3': 1, '4': 1, '5': 1}",0.166667,correct
2,"{'0': 0.2, '1': 0.4, '2': 0.4, '3': 0.4, '4': ...","{'0': 0.0, '1': 0.2, '2': 0.26666666666666666,...","{'0': 0.0, '1': 0.2, '2': 0.30000000000000004,...","{'0': 1, '1': 2, '2': 2, '3': 2, '4': 2, '5': 1}",0,"{'0': {'0': 0, '1': 1, '2': 2, '3': 3, '4': 4,...","{'0': 1.800995096925549e-13, '1': 1.6641194695...","{'0': 0.0607158048829909, '1': 0.1123242344582...","{'0': 0.11143060811761465, '1': 0.480045238476...","{'0': -0.0, '1': 0.11143060811761465, '2': 0.4...","{'0': 1, '1': 1, '2': 1, '3': 1, '4': 1, '5': 1}",0.166667,Hyperparameter
3,"{'0': 1.0, '1': 1.0}","{'0': 0.0, '1': 1.0}","{'0': 0.0, '1': 0.0}","{'0': 1, '1': 1}",0,"{'0': {'0': 0, '1': 1}, '1': {'1': 0}}","{'0': 0.001410435832630306, '1': 0.99999900533...","{'0': 0.3508773619358619, '1': 0.649122638064138}","{'0': 1.0, '1': 0.0}","{'0': 0.0, '1': 1.0}","{'0': 1, '1': 1}",0.500000,correct
4,"{'0': 0.3333333333333333, '1': 0.6666666666666...","{'0': 0.0, '1': 0.3333333333333333, '2': 0.444...","{'0': 0.0, '1': 0.3333333333333333, '2': 0.333...","{'0': 1, '1': 2, '2': 2, '3': 1}",0,"{'0': {'0': 0, '1': 1, '2': 2, '3': 3}, '1': {...","{'0': 9.080114753549862e-09, '1': 7.9178600650...","{'0': 0.11615558155321364, '1': 0.214887855207...","{'0': -1.0243371381948398, '1': 1.295021810905...","{'0': -0.0, '1': -1.0243371381948398, '2': 1.2...","{'0': 1, '1': 1, '2': 1, '3': 1}",0.250000,Hyperparameter
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8157,"{'0': 0.5, '1': 1.0, '2': 0.5}","{'0': 0.0, '1': 0.5, '2': 0.6666666666666666}","{'0': 0.0, '1': 0.5, '2': 0.0}","{'0': 1, '1': 2, '2': 1}",0,"{'0': {'0': 0, '1': 1, '2': 2}, '1': {'1': 0, ...","{'0': 2.9780340102584615e-06, '1': 0.002441987...","{'0': 0.1844167633067642, '1': 0.3411716203526...","{'0': 0.6673223839240217, '1': 0.3326776160759...","{'0': -0.0, '1': 0.6673223839240217, '2': 0.33...","{'0': 1, '1': 1, '2': 1}",0.333333,correct
8158,"{'0': 0.5, '1': 1.0, '2': 0.5}","{'0': 0.0, '1': 0.5, '2': 0.6666666666666666}","{'0': 0.0, '1': 0.5, '2': 0.0}","{'0': 1, '1': 2, '2': 1}",0,"{'0': {'0': 0, '1': 1, '2': 2}, '1': {'1': 0, ...","{'0': 2.9780340102584615e-06, '1': 0.002441987...","{'0': 0.1844167633067642, '1': 0.3411716203526...","{'0': 1.0107604372904642, '1': -0.010760437290...","{'0': -0.0, '1': 1.0107604372904642, '2': -0.0...","{'0': 1, '1': 1, '2': 1}",0.333333,correct
8159,"{'0': 1.0, '1': 1.0}","{'0': 0.0, '1': 1.0}","{'0': 0.0, '1': 0.0}","{'0': 1, '1': 1}",0,"{'0': {'0': 0, '1': 1}, '1': {'1': 0}}","{'0': 0.001410435832630306, '1': 0.99999900533...","{'0': 0.3508773619358619, '1': 0.649122638064138}","{'0': 1.0, '1': 0.0}","{'0': -2.2204460492503136e-16, '1': 1.00000000...","{'0': 1, '1': 1}",0.500000,Hyperparameter
8160,"{'0': 1.0, '1': 1.0}","{'0': 0.0, '1': 1.0}","{'0': 0.0, '1': 0.0}","{'0': 1, '1': 1}",0,"{'0': {'0': 0, '1': 1}, '1': {'1': 0}}","{'0': 0.001410435832630306, '1': 0.99999900533...","{'0': 0.3508773619358619, '1': 0.649122638064138}","{'0': 1.0, '1': -0.0}",

In [5]:
# read csv files as dataframes
codeStructure = pd.read_csv('C:\\Users\\sigma\Documents\\GitHub\ASE_2024\\c_binary_classifiers\\labeled_hyperparameter_codeStructure_padding.csv')
codeStructure

C:\Users\sigma\AppData\Local\Temp\ipykernel_16748\1891209139.py:2: DtypeWarning: Columns (154,155,160,161,171,175,176,177,178,179,180,182,183,184,185,186,187,188,189,190,191,192,194,195,198,199,200,201,204,206,207,210,211,212,213,216,218,219,222,223,224,225,228,230,231,234,235,236,237,238,239,240,242,243,244,245,246,247,248,249,250,251,252,254,255,258,259,260,261,264,266,267,270,271,272,273,274,275,276,278,280,281,282) have mixed types. Specify dtype option on import or set low_memory=False.
  codeStructure = pd.read_csv('C:\\Users\\sigma\Documents\\GitHub\ASE_2024\\c_binary_classifiers\\labeled_hyperparameter_codeStructure_padding.csv')


,0,1,2,3,4,5,6,7,8,9,...,279,280,281,282,283,284,285,286,287,label
0,InputLayer,0.0,NaN,LSTM,NaN,NaN,NaN,"[None, 10, 5]","[None, 10, 8]",False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,correct
1,InputLayer,0.0,NaN,LSTM,NaN,NaN,NaN,"[None, 2, 2]","[None, 100]",False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,correct
2,InputLayer,0.0,NaN,Conv2D,NaN,NaN,NaN,"[None, 100, 100, 1]","[None, 98, 98, 32]",False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,correct
3,InputLayer,0.0,NaN,LSTM,NaN,NaN,NaN,"[None, 10000, 1]","[None, 50]",False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Hyperparameter
4,InputLayer,0.0,NaN,Dense,NaN,NaN,NaN,"[None, 784]","[None, 784]",False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,correct
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8284,InputLayer,0.0,NaN,LSTM,NaN,NaN,NaN,"[None, 1, 1]","[None, 1, 64]",False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Hyperparameter
8285,InputLayer,0.0,NaN,Dense,NaN,NaN,NaN,"[None, 2]","[None, 2]",False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,correct
8286,InputLayer,0.0,NaN,Embedding,NaN,NaN,NaN,"[None, 30]","[None, 30, 300]",False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Hyperparameter
8287,InputLayer,0.0,NaN,Dense,NaN,NaN,NaN,"[None, 2]","[None, 2]",False,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,correct


# Preprocessing the features for fitting into ML clasifier

Embedding

In [14]:
#embedding
data1 = embeddings.copy()
data1['embedding'] = data1['embedding'].apply(lambda x: np.fromstring(x.strip('[]'), sep=', '))
X = np.vstack(data1['embedding'].values)
y = data1['label']
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.2, random_state=42)

In [15]:
# shape of X_train and X_test. Y_train and Y_test
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((6529, 768), (1633, 768), (6529,), (1633,))

In [16]:
# Try all models possible
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import BaggingClassifier
from xgboost import XGBClassifier
from tqdm import tqdm
import xgboost as xgb

def run_models(X_train, X_test, y_train, y_test, verbose=True):
    models = {
        'LogisticRegression': LogisticRegression(max_iter=1000),
        'DecisionTree': DecisionTreeClassifier(),
        'RandomForest': RandomForestClassifier(),
        'SVC': SVC(max_iter=1000),
        'KNeighbors': KNeighborsClassifier(),
        'MLP': MLPClassifier(max_iter=1000),
        'GradientBoosting': GradientBoostingClassifier(),
        'AdaBoost': AdaBoostClassifier(),
        'Bagging': BaggingClassifier(),
        'XGBoost': xgb()
    }
    
    results = {}
    
    for model_name, model in tqdm(models.items(), desc='Training Models', total=len(models)):
        model.fit(X_train, y_train)
        results[model_name] = model.score(X_test, y_test)
        
        if verbose:
            print(model_name, results[model_name])
            
    return results

results = run_models(X_train, X_test, y_train, y_test, verbose=False)
results

Training Models:  30%|███       | 3/10 [00:19<00:57,  8.15s/it]c:\Users\sigma\Documents\GitHub\ASE_2024\myenv\Lib\site-packages\sklearn\svm\_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
Training Models:  70%|███████   | 7/10 [02:35<02:11, 43.89s/it]c:\Users\sigma\Documents\GitHub\ASE_2024\myenv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
Training Models: 100%|██████████| 10/10 [03:37<00:00, 21.72s/it]


{'LogisticRegression': 0.7550520514390692,
 'DecisionTree': 0.5982853643600735,
 'RandomForest': 0.6368646662584201,
 'SVC': 0.5462339252908757,
 'KNeighbors': 0.6883037354562156,
 'MLP': 0.7477036129822413,
 'GradientBoosting': 0.7232088181261482,
 'AdaBoost': 0.7299448867115738,
 'Bagging': 0.6276791181873852,
 'XGBoost': 0.6509491733006736}

Graph

In [18]:
data_graph = graphs.copy()

In [21]:
import pandas as pd
import json
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

def prepare_static_code_graph_dataset(data_graph, verbose=True):
    labels = data_graph['label']

    if verbose:
        print('Number of models:', data_graph.shape[0])
        print('Number of unique labels:', len(set(labels)))

    X_mean = []
    X_median = []
    X_mode = []
    X_max = []
    cols = [col for col in data_graph.columns if col not in ['shortest_path_length', 'label']]
    
    for i in range(data_graph.shape[0]):
        row = data_graph.iloc[i]
        feature_dicts = {}
        for col in cols:
            if col in ['transitivity', 'density']:
                feature_dicts[col] = {0: float(row[col])}
            else:
                try:
                    value = row[col]
                    if isinstance(value, str):
                        value = value.replace("inf", "0").replace("nan", "0")
                        feature_dicts[col] = json.loads(value.replace("'", '"'))
                    else:
                        feature_dicts[col] = {0: 0}
                except (NameError, SyntaxError, json.JSONDecodeError) as e:
                    print(f'Error processing row {i}, column {col}')
                    print('Value:', row[col])
                    print('Error:', e)
                    feature_dicts[col] = {0: 0}
        
        features = [list(feature_dicts[col].values()) for col in cols]
            
        X_mean.append([sum(f)/len(f) for f in features])
        X_median.append([sorted(f)[len(f)//2] for f in features])
        X_mode.append([max(set(f), key=f.count) for f in features])
        X_max.append([max(f) for f in features])
        
    assert len(X_mean) == data_graph.shape[0] == len(labels)

    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(labels)
    
    if verbose:
        print('Columns:', *cols, sep=', ')
        print(f'X.shape: ({len(X_mean)}, {len(X_mean[0])})')
        print('Labels:', *label_encoder.classes_)
        
    return X_mean, X_median, X_mode, X_max, y

X_mean, X_median, X_mode, X_max, y = prepare_static_code_graph_dataset(data_graph)
X_train_mean, X_test_mean, \
X_train_median, X_test_median, \
X_train_mode, X_test_mode, \
X_train_max, X_test_max, \
y_train, y_test = \
train_test_split(X_mean, X_median, X_mode, X_max, y, test_size=0.2, random_state=13)
(X_train_mean, X_test_mean, X_train_median, X_test_median, X_train_mode, X_test_mode, X_train_max, X_test_max, y_train, y_test)

Number of models: 8162
Number of unique labels: 2
Columns:, degree_centrality, closeness_centrality, betweenness_centrality, degree, transitivity, eigenvector_centrality, pagerank, hubs, authorities, k_core, density
X.shape: (8162, 11)
Labels: Hyperparameter correct


([[0.6666666666666666,
   0.38888888888888884,
   0.16666666666666666,
   1.3333333333333333,
   0.0,
   0.33414732808703973,
   0.3333333333333332,
   0.3333333333333333,
   0.3333333333333333,
   1.0,
   0.3333333333333333],
  [0.6666666666666666,
   0.38888888888888884,
   0.16666666666666666,
   1.3333333333333333,
   0.0,
   0.33414732808703973,
   0.3333333333333332,
   0.3333333333333333,
   0.3333333333333333,
   1.0,
   0.3333333333333333],
  [0.3333333333333333,
   0.23666666666666666,
   0.16666666666666666,
   1.6666666666666667,
   0.0,
   0.16757392713768204,
   0.16666666666666663,
   0.16666666666666666,
   0.16666666666666666,
   1.0,
   0.1666666666666666],
  [0.6666666666666666,
   0.38888888888888884,
   0.16666666666666666,
   1.3333333333333333,
   0.0,
   0.33414732808703973,
   0.3333333333333332,
   0.3333333333333333,
   0.3333333333333333,
   1.0,
   0.3333333333333333],
  [0.4,
   0.27166666666666667,
   0.16666666666666666,
   1.6,
   0.0,
   0.200889864424

In [22]:
graph_datasets = {
    'Mean': (X_train_mean, X_test_mean),
    'Median': (X_train_median, X_test_median),
    'Mode': (X_train_mode, X_test_mode),
    'Max': (X_train_max, X_test_max)
}

def run_graph_models(graph_datasets, y_train, y_test, verbose=True):
    results = {}
    for dataset_name, (X_train, X_test) in graph_datasets.items():
        results[dataset_name] = run_models(X_train, X_test, y_train, y_test, verbose)
        
    return results

results = run_graph_models(graph_datasets, y_train, y_test, verbose=False)

from pprint import pprint
for dataset_name, result in results.items():
    print(dataset_name)
    pprint(result)
    
print('Max:', max(max(result.values()) for result in results.values()))

Training Models:  30%|███       | 3/10 [00:00<00:00, 11.76it/s]c:\Users\sigma\Documents\GitHub\ASE_2024\myenv\Lib\site-packages\sklearn\svm\_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
Training Models:  70%|███████   | 7/10 [00:02<00:01,  2.27it/s]c:\Users\sigma\Documents\GitHub\ASE_2024\myenv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
Training Models:  30%|███       | 3/10 [00:00<00:01,  3.73it/s]c:\Users\sigma\Documents\GitHub\ASE_2024\myenv\Lib\site-packages\sklearn\svm\_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
Training Models:  70%|███████   | 7/10 [00:03<00:01,  1.91it

Mean
{'AdaBoost': 0.5664421310471525,
 'Bagging': 0.5578689528475199,
 'DecisionTree': 0.5664421310471525,
 'GradientBoosting': 0.5664421310471525,
 'KNeighbors': 0.5229638701775873,
 'LogisticRegression': 0.5315370483772198,
 'MLP': 0.5315370483772198,
 'RandomForest': 0.5664421310471525,
 'SVC': 0.4947948560930802,
 'XGBoost': 0.5664421310471525}
Median
{'AdaBoost': 0.5682792406613595,
 'Bagging': 0.5070422535211268,
 'DecisionTree': 0.5107164727495407,
 'GradientBoosting': 0.5621555419473362,
 'KNeighbors': 0.5107164727495407,
 'LogisticRegression': 0.48193508879363134,
 'MLP': 0.554194733619106,
 'RandomForest': 0.5094917330067361,
 'SVC': 0.49785670545009186,
 'XGBoost': 0.545009185548071}
Mode
{'AdaBoost': 0.5456215554194733,
 'Bagging': 0.5156154317207593,
 'DecisionTree': 0.5137783221065524,
 'GradientBoosting': 0.5437844458052664,
 'KNeighbors': 0.5290875688916106,
 'LogisticRegression': 0.5266380894060012,
 'MLP': 0.5376607470912431,
 'RandomForest': 0.5137783221065524,
 'SVC

Local Code Structue

In [25]:
file_path_code_structure = 'C:\\Users\\sigma\\Documents\\GitHub\\ASE_2024\\c_binary_classifiers\\labeled_hyperparameter_codeStructure_padding.csv'
data = pd.read_csv(file_path_code_structure)

data = data.fillna(0)
X = data.drop(columns=['label'])
X = pd.get_dummies(X, drop_first=True)
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(data['label'])
scaler = StandardScaler()
X_normalized = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(X_normalized, y, test_size=0.2, random_state=42)

C:\Users\sigma\AppData\Local\Temp\ipykernel_16748\373761355.py:2: DtypeWarning: Columns (154,155,160,161,171,175,176,177,178,179,180,182,183,184,185,186,187,188,189,190,191,192,194,195,198,199,200,201,204,206,207,210,211,212,213,216,218,219,222,223,224,225,228,230,231,234,235,236,237,238,239,240,242,243,244,245,246,247,248,249,250,251,252,254,255,258,259,260,261,264,266,267,270,271,272,273,274,275,276,278,280,281,282) have mixed types. Specify dtype option on import or set low_memory=False.
  data = pd.read_csv(file_path_code_structure)


In [33]:
import xgboost as xgb
def run_models(X_train, X_test, y_train, y_test, verbose=True):
    models = {
        'LogisticRegression': LogisticRegression(max_iter=1000),
        'DecisionTree': DecisionTreeClassifier(),
        'RandomForest': RandomForestClassifier(),
        'SVC': SVC(max_iter=1000),
        'KNeighbors': KNeighborsClassifier(),
        'MLP': MLPClassifier(max_iter=1000),
        'GradientBoosting': GradientBoostingClassifier(),
        'AdaBoost': AdaBoostClassifier(),
        'Bagging': BaggingClassifier(),
        'XGBoost': xgb.XGBClassifier()
    }
    
    results = {}
    
    for model_name, model in tqdm(models.items(), desc='Training Models', total=len(models)):
        model.fit(X_train, y_train)
        results[model_name] = model.score(X_test, y_test)
        
        if verbose:
            print(model_name, results[model_name])
            
    return results

In [34]:
# Run the models
results = run_models(X_train, X_test, y_train, y_test, verbose=False)
results

Training Models:  30%|███       | 3/10 [00:05<00:15,  2.20s/it]c:\Users\sigma\Documents\GitHub\ASE_2024\myenv\Lib\site-packages\sklearn\svm\_base.py:297: ConvergenceWarning: Solver terminated early (max_iter=1000).  Consider pre-processing your data with StandardScaler or MinMaxScaler.
  warnings.warn(
Training Models:  70%|███████   | 7/10 [00:52<00:36, 12.29s/it]c:\Users\sigma\Documents\GitHub\ASE_2024\myenv\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
Training Models: 100%|██████████| 10/10 [01:07<00:00,  6.72s/it]


{'LogisticRegression': 0.7237635705669482,
 'DecisionTree': 0.7243667068757539,
 'RandomForest': 0.7243667068757539,
 'SVC': 0.5150784077201448,
 'KNeighbors': 0.6682750301568154,
 'MLP': 0.7195416164053076,
 'GradientBoosting': 0.7195416164053076,
 'AdaBoost': 0.7243667068757539,
 'Bagging': 0.7195416164053076,
 'XGBoost': 0.7243667068757539}

In [36]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report

clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1, oob_score=True)
clf.fit(X_train, y_train)

importances = clf.feature_importances_
indices = np.argsort(importances)[::-1]
k = 20
X_train_selected = X_train[:, indices[:k]]
X_test_selected = X_test[:, indices[:k]]

print("Feature ranking:")
for f in range(X_train_selected.shape[1]):
    print("%d. feature %d (%f)" % (f + 1, indices[f], importances[indices[f]]))

feature_names = X.columns
feature_names = feature_names[indices]
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],  
    'max_features': ['sqrt', 'log2'] 
}

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42, n_jobs=-1, oob_score=True),
    param_grid=param_grid,
    cv=5, 
    verbose=2,
    n_jobs=-1
)
grid_search.fit(X_train_selected, y_train)
best_clf = grid_search.best_estimator_

y_pred = best_clf.predict(X_test_selected)
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))
print("OOB Score of Best Model: ", best_clf.oob_score_)

Feature ranking:
1. feature 336 (0.025993)
2. feature 473 (0.022606)
3. feature 255 (0.018302)
4. feature 331 (0.016754)
5. feature 392 (0.013222)
6. feature 168 (0.011352)
7. feature 205 (0.009714)
8. feature 318 (0.009412)
9. feature 395 (0.009072)
10. feature 184 (0.008932)
11. feature 382 (0.008425)
12. feature 55 (0.007765)
13. feature 220 (0.007571)
14. feature 672 (0.007333)
15. feature 276 (0.007022)
16. feature 300 (0.006993)
17. feature 779 (0.006884)
18. feature 118 (0.006768)
19. feature 272 (0.006603)
20. feature 142 (0.006464)
Fitting 5 folds for each of 216 candidates, totalling 1080 fits
                precision    recall  f1-score   support

Hyperparameter       0.57      0.99      0.73       838
       correct       0.95      0.25      0.40       820

      accuracy                           0.62      1658
     macro avg       0.76      0.62      0.56      1658
  weighted avg       0.76      0.62      0.56      1658

OOB Score of Best Model:  0.6281103905896547
